### Dependencies

In [83]:
import math
import random

### Micrograd

In [84]:
class Value:
    def __init__(self, data: float, _children=(), _op="", label=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward

        return out

    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward

        return out

    def __rmul__(self, other):
        return self * other

    def __neg__(self):  # -self
        return self * -1

    def __sub__(self, other):  # self - other
        return self + (-other)

    def __rsub__(self, other):  # other - self
        return other + (-self)

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other, (self,), f"**{other}")

        def _backward():
            self.grad += other * self.data ** (other - 1) * out.grad

        out._backward = _backward

        return out

    def __truediv__(self, other):
        return self * other**-1

    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward

        return out

    def tanh(self):
        out = Value(
            (math.exp(2 * self.data) - 1) / (math.exp(2 * self.data) + 1),
            (self,),
            "tanh",
        )

        def _backward():
            self.grad += (1 - out.data**2) * out.grad

        out._backward = _backward

        return out

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

### Neural Network

In [85]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

In [86]:
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

In [87]:
class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

### Training

In [88]:
xs = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [5.0, 1.0, 1.0], [1.0, 1.0, -1.0]]

ys = [1.0, -1.0, -1.0, 1.0]

In [89]:
n = MLP(len(xs), [4, 4, 1])

In [90]:
for k in range(1000):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt) ** 2 for yout, ygt in zip(ypred, ys))

    # backward pass
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()

    # upddate (gradient descent)
    for p in n.parameters():
        p.data -= 0.01 * p.grad

    print(f"Iteration {k}, Loss: {loss.data}")

Iteration 0, Loss: 2.492671626617253
Iteration 1, Loss: 2.1337540747870207
Iteration 2, Loss: 1.8671547812952762
Iteration 3, Loss: 1.6454012870021884
Iteration 4, Loss: 1.4526739647315297
Iteration 5, Loss: 1.2842384148120982
Iteration 6, Loss: 1.1379276669160374
Iteration 7, Loss: 1.0117048182741524
Iteration 8, Loss: 0.9032831396059653
Iteration 9, Loss: 0.8102817178903952
Iteration 10, Loss: 0.7304249141089784
Iteration 11, Loss: 0.6616657250217319
Iteration 12, Loss: 0.6022332770465106
Iteration 13, Loss: 0.5506316195140037
Iteration 14, Loss: 0.505614108734088
Iteration 15, Loss: 0.4661489544391708
Iteration 16, Loss: 0.4313842770610752
Iteration 17, Loss: 0.40061645589965306
Iteration 18, Loss: 0.3732630419296507
Iteration 19, Loss: 0.34884028333750594
Iteration 20, Loss: 0.326944793464758
Iteration 21, Loss: 0.30723873050666284
Iteration 22, Loss: 0.2894378633888739
Iteration 23, Loss: 0.27330196946292573
Iteration 24, Loss: 0.25862709911050585
Iteration 25, Loss: 0.24523932879

In [91]:
ypred

[Value(data=0.9847183404662863),
 Value(data=-0.9717310487158369),
 Value(data=-0.9716371207689631),
 Value(data=0.9670146746668103)]